In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2501747592_7593.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/1000827867_1000827869.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2505116322.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2504046077.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2024493246-a.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2055626759.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2082740426.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/scientific_report/2505473647_3650.tif
/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/val/sc

In [10]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib

## Task 1.1: Load and Prepare Data

## Task 1.2: Train Classifier

In [11]:
import os
import pytesseract
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib


def load_documents(data_dir):
    """
    Load documents from folder structure and extract text using OCR.

    Folder structure example:
    training_data/
        invoices/
            img1.png
            img2.png
        receipts/
            img3.png
    """

    documents = []
    labels = []

    # Loop through each document type folder
    for doc_type in os.listdir(data_dir):

        folder_path = os.path.join(data_dir, doc_type)

        # Skip if not a folder
        if not os.path.isdir(folder_path):
            continue

        # Loop through images inside folder
        for filename in os.listdir(folder_path):

            file_path = os.path.join(folder_path, filename)

            try:
                # Open image
                img = Image.open(file_path)

                # Extract text using OCR
                text = pytesseract.image_to_string(img)

                # Store text and label
                documents.append(text)
                labels.append(doc_type)

            except Exception as e:
                print(f"Error processing {file_path}: {e}")

    return documents, labels


# ================= LOAD DATA =================
data_directory = "/kaggle/input/datasets/vaclavpechtor/rvl-cdip-small-200/rvl-cdip-small-200/train"

documents, labels = load_documents(data_directory)

print(f"Loaded {len(documents)} documents")
print(f"Classes: {set(labels)}")


# ================= TEXT VECTORIZATION =================
vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(documents)
y = labels


# ================= TRAIN / TEST SPLIT =================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ================= MODEL TRAINING =================
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)


# ================= PREDICTION =================
y_pred = model.predict(X_test)


# ================= EVALUATION =================
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# ================= SAVE MODEL =================
joblib.dump(model, "document_classifier.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("\nModel and vectorizer saved successfully!")

Loaded 2560 documents
Classes: {'letter', 'memo', 'form', 'scientific_publication', 'questionnaire', 'presentation', 'email', 'scientific_report', 'handwritten', 'file_folder', 'invoice', 'resume', 'advertisement', 'specification', 'news_article', 'budget'}

Accuracy: 0.6250

Classification Report:
                        precision    recall  f1-score   support

         advertisement       0.55      0.42      0.48        26
                budget       0.74      0.56      0.63        36
                 email       0.97      0.88      0.92        33
           file_folder       0.40      0.84      0.54        38
                  form       0.74      0.42      0.54        33
           handwritten       0.34      0.44      0.38        34
               invoice       0.72      0.67      0.69        27
                letter       0.61      0.61      0.61        36
                  memo       0.73      0.54      0.62        35
          news_article       0.38      0.57      0.46      

## Task 1.3: Save Model

In [12]:
# ================= SAVE TRAINED MODELS =================

import os
import joblib

# Create folder if it does not exist
os.makedirs("models", exist_ok=True)

# Save trained vectorizer and classifier
joblib.dump(vectorizer, "models/vectorizer.pkl")
joblib.dump(model, "models/document_classifier.pkl")

print("Models saved successfully!")

# ================= LOAD SAVED MODELS =================

loaded_vectorizer = joblib.load("models/vectorizer.pkl")
loaded_classifier = joblib.load("models/document_classifier.pkl")

print("Models loaded successfully!")

# ================= TEST THE LOADED MODELS =================

sample_text = [
    "Invoice Number INV-1001 Total Amount 450 USD Vendor Amazon Date 12-05-2026"
]

# Convert text into TF-IDF features
sample_features = loaded_vectorizer.transform(sample_text)

# Predict document class
prediction = loaded_classifier.predict(sample_features)

print(f"\nPredicted Document Type: {prediction[0]}")

Models saved successfully!
Models loaded successfully!

Predicted Document Type: invoice


# PART 2: BUILD REST API

## Task 2.1: Create FastAPI App

In [13]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from PIL import Image
import pytesseract
import joblib
import io
import os

# ================= INITIALIZE FASTAPI =================

app = FastAPI(
    title="Document Intelligence API",
    description="OCR, Classification, and Extraction API",
    version="1.0.0"
)

# ================= LOAD TRAINED MODELS =================

vectorizer = joblib.load("models/vectorizer.pkl")
classifier = joblib.load("models/document_classifier.pkl")

print("Models loaded successfully!")

# ================= ROOT ENDPOINT =================

@app.get("/")
def root():
    return {
        "message": "Document Intelligence API",
        "version": "1.0.0",
        "endpoints": [
            "/classify",
            "/extract",
            "/process"
        ]
    }

# ================= CLASSIFICATION ENDPOINT =================

@app.post("/classify")
async def classify_document(file: UploadFile = File(...)):

    try:
        # Read uploaded image
        image_bytes = await file.read()

        # Convert bytes to image
        image = Image.open(io.BytesIO(image_bytes))

        # OCR Extraction
        extracted_text = pytesseract.image_to_string(image)

        # Convert text into TF-IDF features
        features = vectorizer.transform([extracted_text])

        # Predict document type
        prediction = classifier.predict(features)[0]

        return JSONResponse(content={
            "filename": file.filename,
            "predicted_class": prediction,
            "extracted_text": extracted_text
        })

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": str(e)}
        )

Models loaded successfully!


## Task 2.3: Add Extraction Endpoint

In [21]:
# ================= IMPORT EXTRACTION FUNCTIONS =================

import sys
import io
from fastapi import UploadFile, File
from fastapi.responses import JSONResponse
from PIL import Image
import pytesseract

# Add parent directory to Python path
sys.path.append("../")

# Import custom extraction functions
from week7_information_extraction import (
    extract_dates,
    extract_amounts,
    extract_entities
)

# ================= INFORMATION EXTRACTION ENDPOINT =================

@app.post("/extract")
async def extract_information(file: UploadFile = File(...)):
    """
    Extract dates, amounts, and entities from uploaded document.
    """

    try:
        # Read uploaded file
        contents = await file.read()

        # Convert bytes into image
        image = Image.open(io.BytesIO(contents))

        # OCR text extraction
        text = pytesseract.image_to_string(image)

        # ================= EXTRACT INFORMATION =================

        dates = extract_dates(text)
        amounts = extract_amounts(text)
        entities = extract_entities(text)

        # ================= RETURN RESULTS =================

        return {
            "filename": file.filename,
            "dates": dates,
            "amounts": amounts,
            "entities": entities,
            "raw_text_preview": text[:500]   # First 500 characters
        }

    except Exception as e:

        return JSONResponse(
            status_code=500,
            content={
                "error": str(e)
            }
        )

ModuleNotFoundError: No module named 'week7_information_extraction'